# Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import glob
import graphical_sampling as gs
import pandas as pd
import numpy as np
import itertools
from tqdm import tqdm
from package_sampling.utils import inclusion_probabilities

/home/divar/projects/graphical-sampling/.venv/lib/python3.12/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "LD_LIBRARY_PATH" redefined by R and overriding existing variable. Current: "/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server", R: "/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server:/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server"
  warnings.warn(
/home/divar/projects/graphical-sampling/.venv/lib/python3.12/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_LIBS_SITE" redefined by R and overriding existing variable. Current: "/usr/local/lib/R/site-library/:/usr/local/lib/R/site-library:/usr/lib/R/site-library:/usr/lib/R/library", R: "/usr/local/lib/R/site-library/:/usr/local/lib/R/site-library/:/usr/local/lib/R/site-library/:/usr/local/lib/R/site-library:/usr/lib/R/site-library:/usr/lib/R/library:/usr/lib/R/library:/usr/lib/R/

# Loading and Determining Population

In [3]:
DATA_DIR = "populations"
csv_paths = glob.glob(os.path.join(DATA_DIR, "*.csv"))

coords_dict = {}
probs_dict = {}

for fp in csv_paths:
    name = os.path.splitext(os.path.basename(fp))[0]
    data = np.loadtxt(fp, delimiter=",", skiprows=1)
    coords = data[:, :2]
    probs  = data[:, -1]

    coord_name, prob_name, *rest = name.split("_")
    coord_name = 'cluster' if coord_name == 'clust' else coord_name
    prob_name = 'equal' if prob_name == 'eq' else 'unequal'

    coords_dict[coord_name] = coords
    probs_dict[coord_name] = probs_dict.get(coord_name, {})
    probs_dict[coord_name][prob_name] = probs

print(coords_dict.keys())
print(probs_dict.keys())
print(probs_dict['random'].keys())

dict_keys(['cluster', 'AggregatedPop1027', 'random', 'meuse', 'grid', 'RegularPop1000', 'swiss'])
dict_keys(['cluster', 'AggregatedPop1027', 'random', 'meuse', 'grid', 'RegularPop1000', 'swiss'])
dict_keys(['equal', 'unequal'])


In [37]:
coords = np.loadtxt("populations/AggregatedPop1027_2d.csv", delimiter=",", skiprows=0)
# coords = np.loadtxt("populations/RegularPop1000_2d.csv", delimiter=",", skiprows=0)
print(len(coords))
rng = gs.random.rng()
n = 50
probs = rng.equal_probabilities(n, coords.shape[0])
pop = gs.Population(coords, probs)


1027


In [ ]:
# n = 20, spiral spiral Regular
# density	0.006099	0.030789
# moran	-0.254578	0.043766
# local_balance	0.129771	0.040084
# voronoi	0.076617	0.039016

# n = 50, spiral spiral Regular
# density	0.004681	0.013391
# moran	-0.356249	0.027877
# local_balance	0.070798	0.015639
# voronoi	0.060610	0.022326

# n = 100, spiral spiral Regular
# density	0.003941	0.010907
# moran	-0.422802	0.028637
# local_balance	0.048174	0.005673
# voronoi	0.061820	0.010714


# n = 20, spiral spiral Aggregated
# density	0.070551	0.059996
# moran	-0.272021	0.037297
# local_balance	0.117540	0.021086
# voronoi	0.088505	0.031076


density	0.021019	0.030379
moran	-0.383385	0.033951
local_balance	0.085965	0.009068
voronoi	0.115908	0.021655


# Building Initial Designs

In [38]:
orders = [
    # "lexico-yx",
    # "lexico-xy",
    # "random",
    # "angle_0",
    # "distance_0",
    # "projection",
    # "center",
    "spiral",
    # "max",
    # "snake",
    # "hilbert",
]
n_zones_sweep = [(2, 2)]
# n_zones_cluster = [4]
combinations = list(itertools.product(['sweep'], n_zones_sweep, orders, orders))# + list(itertools.product(['cluster'], n_zones_cluster, orders, orders))

In [39]:
initial_designs = []
num_trials = 1
for zone_builder, n_zones, units_order, zones_order in tqdm(combinations, desc="Generating initial designs", total=len(combinations), unit="orders"):
    best = None
    best_score = np.inf
    for _ in range(num_trials):
        ks = gs.sampling.KMeansSampler(
            population=pop,
            n=n,
            n_zones=n_zones,
            zone_builder=zone_builder,
            units_order=units_order,
            zones_order=zones_order,
            split_size=0.05
        )
        if ks.expected_moran_score() < best_score:
            best = ks
            best_score = ks.expected_moran_score()
            print(f"New best score: {best_score} for {zone_builder}, {n_zones}, {units_order}, {zones_order}")

    initial_designs.append(gs.NewDesign(best))

Generating initial designs:   0%|          | 0/1 [00:00<?, ?orders/s]

/home/divar/projects/graphical-sampling/.venv/lib/python3.12/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "LD_LIBRARY_PATH" redefined by R and overriding existing variable. Current: "/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server:/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server", R: "/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server:/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server:/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server"
  warnings.warn(
/home/divar/projects/graphical-sampling/.venv/lib/python3.12/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_LIBS_SITE" redefined by R and overriding existing variable. Current: "/usr/local/lib/R/site-library/:/usr/local/lib/R/site-library/:/usr/local/lib/R/site-library/:/usr/local/lib/R/site-library:/usr/lib/R/site-library:/us

New best score: -0.29240916395542116 for sweep, (2, 2), spiral, spiral


# Run

In [41]:
moran_criteria = gs.criteria.MoranCriteria()

In [42]:
initial_criteria_value = np.array([moran_criteria(design) for design in initial_designs])
print(initial_criteria_value)
best_design = initial_designs[np.argmin(initial_criteria_value)]
best_criteria_value = np.min(initial_criteria_value)
best_criteria_value
initial_designs[np.argmin(initial_criteria_value)].kmeans.score_summary_df()


[-0.29240916]


,expected,std
measure,,
density,0.016866,0.020760
moran,-0.292409,0.059658
local_balance,0.089572,0.013403
voronoi,0.122967,0.034544


In [43]:
astar = gs.search.AStar(
    initial_designs,
    moran_criteria
)

best initial criteria value -0.29240916395542116


In [36]:
astar = gs.search.AStar(
    [astar.best_design],
    moran_criteria
)

best initial criteria value -0.4154389025088517


In [44]:
astar.run(
    max_iterations = 100,
    num_new_nodes = 20,
    max_open_set_size = 1000,

    n_clusters_to_change_order_zone = 1,
    n_changes_in_order_of_zones = 1,

    n_clusters_to_change_order_units = 1,
    n_zones_to_change_order_units = 1,
    n_changes_in_order_of_units = 1,

    n_jobs=-1
)


parent node: -0.29240916395542116


/home/divar/projects/graphical-sampling/.venv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/home/divar/projects/graphical-sampling/.venv/lib/python3.12/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "LD_LIBRARY_PATH" redefined by R and overriding existing variable. Current: "/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server:/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server", R: "/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server:/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server:/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server"
  warnings.warn(
/home/divar/projects/graphical-sampling/.venv/lib/python3.12/site-packages/rpy2/rin

child node: -0.2939571916701132

New best criteria value: -0.2939571916701132
child node: -0.28747219279268504
child node: -0.2866496968716242
child node: -0.28631175809708215
child node: -0.29108181402511835
child node: -0.2961172417729866

New best criteria value: -0.2961172417729866
child node: -0.2947270456306546
child node: -0.29139952341853265
child node: -0.2896439567534284
child node: -0.2909342271080893
child node: -0.29240916395542116
child node: -0.2907773389810136
child node: -0.2861447759067601
child node: -0.29240009519890986
child node: -0.2923204433087715
child node: -0.29240942105914525
child node: -0.2888861663393187
child node: -0.2908423868651557
child node: -0.292305503702731
child node: -0.29084974276259107

parent node: -0.2961172417729866


/home/divar/projects/graphical-sampling/.venv/lib/python3.12/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "LD_LIBRARY_PATH" redefined by R and overriding existing variable. Current: "/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server:/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server", R: "/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server:/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server:/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server"
  warnings.warn(
/home/divar/projects/graphical-sampling/.venv/lib/python3.12/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_LIBS_SITE" redefined by R and overriding existing variable. Current: "/usr/local/lib/R/site-library/:/usr/local/lib/R/site-library/:/usr/local/lib/R/site-library/:/usr/local/lib/R/site-library:/usr/lib/R/site-library:/us

child node: -0.29417371301798567
child node: -0.29711210035437935

New best criteria value: -0.29711210035437935
child node: -0.29608495476892865
child node: -0.29564383049141746
child node: -0.2953274476587322
child node: -0.29564879540569416
child node: -0.28912311577611016
child node: -0.29137228805180765
child node: -0.292056359959373
child node: -0.2966601347311457
child node: -0.28857136877886225
child node: -0.2916301964768923
child node: -0.2901158069756906
child node: -0.29452157423201025
child node: -0.29624443113518784
child node: -0.2919436831285865
child node: -0.294932918736441
child node: -0.29455723824251284
child node: -0.2901536638365574
child node: -0.2960752940107042

parent node: -0.29711210035437935
child node: -0.2913702653256981
child node: -0.29713867225466073

New best criteria value: -0.29713867225466073
child node: -0.3008536111638138

New best criteria value: -0.3008536111638138
child node: -0.29615282860870656
child node: -0.29545813476416743
child node: -

100

In [45]:
astar.best_design.kmeans.score_summary_df()

,expected,std
measure,,
density,0.021019,0.030379
moran,-0.383385,0.033951
local_balance,0.085965,0.009068
voronoi,0.115908,0.021655


In [23]:
astar.best_criteria_value

-0.24898385729914374

In [24]:
astar.best_design.kmeans.all_samples.shape

(332, 20)

In [25]:
np.sum(astar.best_design.kmeans.all_samples_probs)

np.float64(1.0)

In [27]:
np.mean(np.abs(astar.best_design.kmeans.fips - probs))

np.float64(0.0006219611368455774)

In [28]:
astar.best_design.kmeans.fips

array([0.0194742, 0.0194742, 0.0194742, ..., 0.0194742, 0.0194742,
       0.0194742], shape=(1027,))

In [29]:
np.sqrt(astar.best_design.kmeans.var_moran_score())

np.float64(0.051920976433342605)

In [76]:
astar.best_design.kmeans.moran_scores

array([-0.50854305, -0.54293679, -0.65922451, -0.64147852, -0.57183288,
       -0.58202557, -0.56105458, -0.45920847, -0.5618256 , -0.59335246,
       -0.54823247, -0.589926  , -0.53048317, -0.56353487, -0.54957034,
       -0.55135462, -0.63054903, -0.62841727, -0.59349016, -0.71381491,
       -0.66701377, -0.57776011, -0.55605104, -0.58317301, -0.63929135,
       -0.48359111, -0.61539171, -0.51574499, -0.55439157, -0.58472678,
       -0.71808128, -0.6459911 , -0.52057374, -0.59028964, -0.57343073,
       -0.61691472, -0.70571138, -0.56483565, -0.64364067, -0.60274384,
       -0.62059921, -0.57920348, -0.71291035, -0.54268658, -0.63107332,
       -0.52090799, -0.65304167, -0.68789819, -0.45113366, -0.39102337])

In [67]:
astar.best_design.kmeans.score_summary_df()

,expected,std
measure,,
density,0.166571,0.117680
moran,-0.452472,0.068235
local_balance,0.226800,0.052961
voronoi,0.090396,0.046718


NameError: name 'initial_design' is not defined

In [30]:
def info(kmeans):
    mean = round(float(kmeans.expected_moran_score()), 2)
    moran_scores = astar.best_design.kmeans.moran_scores
    median = round(float(np.median(moran_scores)), 2)
    sd = round(float(np.sqrt(kmeans.var_moran_score())), 2)
    min_ = round(float(np.min(moran_scores)), 2)
    max_ = round(float(np.max(moran_scores)), 2)
    return mean, median, sd, min_, max_

In [31]:
info(astar.best_design.kmeans)

(-0.25, -0.25, 0.05, -0.4, -0.14)

In [23]:
gs.plot(pop, astar.best_design.kmeans.clusters, connect_points=False)

NameError: name 'astar' is not defined